In [1]:

import torch ## torch let's us create tensors and also provides helper functions
import torch.nn as nn ## torch.nn gives us nn.Module(), nn.Embedding() and nn.Linear()
import torch.nn.functional as F # This gives us the softmax() and argmax()
from torch.optim import Adam ## We will use the Adam optimizer, which is, essentially,
                             ## a slightly less stochastic version of stochastic gradient descent.
from torch.utils.data import TensorDataset, DataLoader ## We'll store our data in DataLoaders

import lightning as L

In [2]:
## first, we create a dictionary that maps vocabulary tokens to id numbers...
english_token_to_id = {'lets': 0,
                       'to': 1,
                       'go': 2,
                       '<EOS>': 3 ## <EOS> = end of sequence
                      }
## ...then we create a dictionary that maps the ids to tokens. This will help us interpret the output.
## We use the "map()" function to apply the "reversed()" function to each tuple (i.e. ('lets', 0)) stored
## in the token_to_id dictionary. We then use dict() to make a new dictionary from the
## reversed tuples.
english_id_to_token = dict(map(reversed, english_token_to_id.items()))

spanish_token_to_id = {'ir': 0,
                       'vamos': 1,
                       'y': 2,
                       '<EOS>': 3}
spanish_id_to_token = dict(map(reversed, spanish_token_to_id.items()))

inputs = torch.tensor([[english_token_to_id["lets"],
                        english_token_to_id["go"]],

                       [english_token_to_id["to"],
                        english_token_to_id["go"]]])

labels = torch.tensor([[spanish_token_to_id["vamos"],
                        spanish_token_to_id["<EOS>"]],

                       [spanish_token_to_id["ir"],
                        spanish_token_to_id["<EOS>"]]])

In [3]:
dataset = TensorDataset(inputs, labels)
dataloader = DataLoader(dataset)

## Build and Train a Seq2Seq/Encoder-Decoder Model with Attention from Scratch

In [26]:
class Seq2SeqAttention(L.LightningModule):

    def __init__(self, max_len=2):
        super().__init__()

        # maximum number of tokens decoder can generate
        self.max_decoder_length = max_len

        # reproducibility
        L.seed_everything(42)

        #################################
        # ENCODER
        #################################

        # token → embedding
        self.encoder_embedding = nn.Embedding(
            num_embeddings=4,
            embedding_dim=1
        )

        # processes input sequence
        self.encoder_lstm = nn.LSTM(
            input_size=1,
            hidden_size=1,
            num_layers=1
        )

        #################################
        # DECODER
        #################################

        # token → embedding
        self.decoder_embedding = nn.Embedding(
            num_embeddings=4,
            embedding_dim=1
        )

        # generates output sequence
        self.decoder_lstm = nn.LSTM(
            input_size=1,
            hidden_size=1,
            num_layers=1
        )

        # combines:
        # (attention context + decoder output)
        self.output_layer = nn.Linear(
            in_features=2,   # [attention_value, decoder_hidden]
            out_features=4   # vocab size
        )

        self.loss_fn = nn.CrossEntropyLoss()


    def forward(self, input_tokens, target_tokens=None):

        #################################
        # 1. ENCODER
        #################################

        # convert tokens → embeddings
        encoder_embeds = self.encoder_embedding(input_tokens)

        # pass through LSTM
        # encoder_outputs = hidden state at EVERY time step
        # final_hidden, final_cell = summary of sequence
        encoder_outputs, (encoder_hidden, encoder_cell) = self.encoder_lstm(encoder_embeds)

        #################################
        # 2. INITIALIZE DECODER
        #################################

        # start token (<EOS> used as start)
        current_token_id = torch.tensor([spanish_token_to_id["<EOS>"]])

        # embed start token
        decoder_embeds = self.decoder_embedding(current_token_id)

        # initialize decoder with encoder's final states
        decoder_output, (decoder_hidden, decoder_cell) = self.decoder_lstm(
            decoder_embeds, (encoder_hidden, encoder_cell)
        )

        #################################
        # 3. ATTENTION (FIRST STEP)
        #################################

        # compute similarity between:
        # current decoder state and ALL encoder outputs
        # (dot product attention)
        similarity_scores = torch.matmul(
            decoder_output,
            encoder_outputs.transpose(0, 1)
        )

        # convert scores → probabilities
        attention_weights = F.softmax(similarity_scores, dim=1)

        # weighted sum of encoder outputs
        attention_context = torch.matmul(attention_weights, encoder_outputs)

        # combine context + decoder state
        combined_vector = torch.cat(
            (attention_context, decoder_output),
            dim=1
        )

        # project to vocabulary
        logits = self.output_layer(combined_vector)

        all_outputs = logits

        # greedy prediction
        predicted_token_id = torch.tensor([torch.argmax(logits)])

        #################################
        # 4. DECODING LOOP
        #################################

        for t in range(1, self.max_decoder_length):

            if target_tokens is None:
                # inference mode

                # stop if <EOS>
                if predicted_token_id == spanish_token_to_id["<EOS>"]:
                    break

                next_input_id = predicted_token_id

            else:
                # training mode (teacher forcing)
                next_input_id = torch.tensor([target_tokens[t - 1]])

            # embed next token
            decoder_embeds = self.decoder_embedding(next_input_id)

            # pass through decoder LSTM
            decoder_output, (decoder_hidden, decoder_cell) = self.decoder_lstm(
                decoder_embeds, (decoder_hidden, decoder_cell)
            )

            #################################
            # ATTENTION (EVERY STEP)
            #################################

            similarity_scores = torch.matmul(
                decoder_output,
                encoder_outputs.transpose(0, 1)
            )

            attention_weights = F.softmax(similarity_scores, dim=1)

            attention_context = torch.matmul(attention_weights, encoder_outputs)

            combined_vector = torch.cat(
                (attention_context, decoder_output),
                dim=1
            )

            logits = self.output_layer(combined_vector)

            # store outputs
            all_outputs = torch.cat((all_outputs, logits), dim=0)

            # next prediction
            predicted_token_id = torch.tensor([torch.argmax(logits)])

        return all_outputs


    def configure_optimizers(self):
        return Adam(self.parameters(), lr=0.1)


    def training_step(self, batch, batch_idx):

        input_tokens, target_tokens = batch

        logits = self.forward(input_tokens[0], target_tokens[0])

        loss = self.loss_fn(logits, target_tokens[0])

        return loss

In [27]:

model = seq2seq_attention()
outputs = model.forward(input_tokens=torch.tensor([english_token_to_id["lets"],
                                            english_token_to_id["go"]]), ## translate "lets go", we should get "vamos <EOS>"
                        target_tokens=None)

print("Translated text:")
predicted_ids = torch.argmax(outputs, dim=1)
for id in predicted_ids:
    print("\t", spanish_id_to_token[id.item()])

Seed set to 42


Translated text:
	 y
	 y


In [28]:
trainer = L.Trainer(max_epochs=20, accelerator="cpu")
trainer.fit(model, train_dataloaders=dataloader)

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.

  | Name              | Type             | Params | Mode  | FLOPs
-----------------------------------------------------------------------
0 | encoder_embedding | Embedding        | 4      | train | 0    
1 | encoder_lstm      | LSTM             | 16     | train | 0    
2 | decoder_embedding | Embedding        | 4      | train | 0    
3 | decoder_lstm      | LSTM             | 16     | train | 0    
4 | output_layer      | Linear           | 12     | train | 0    
5 | loss        

Epoch 19: 100%|█| 2/2 [00:00<00:00, 497.40it/s

`Trainer.fit` stopped: `max_epochs=20` reached.


Epoch 19: 100%|█| 2/2 [00:00<00:00, 311.81it/s


In [29]:
outputs = model.forward(input_tokens=torch.tensor([english_token_to_id["to"],
                                            english_token_to_id["go"]]), ## translate "lets go", we should get "vamos <EOS>"
                        target_tokens=None)

print("Translated text:")
predicted_ids = torch.argmax(outputs, dim=1)
for id in predicted_ids:
    print("\t", spanish_id_to_token[id.item()])

Translated text:
	 ir
	 <EOS>




Continuing Chapter 11 of The StatQuest Illustrated Guide to Neural Networks and AI, I explored how attention improves Sequence-to-Sequence models.

In the previous chapter, Seq2Seq models relied on a single fixed-size vector to represent the entire input sequence. This created a major limitation: as input length increases, important information gets compressed and lost.

Attention addresses this problem by allowing the model to look back at the entire input sequence dynamically during decoding.

⸻

Core Idea

Instead of forcing the encoder to compress everything into one vector, attention allows the decoder to:

focus on different parts of the input at each step

So the model no longer depends on a single summary. It selectively retrieves relevant information when needed.

⸻

Key Mechanism

At every decoding step, the model performs three operations:

1. Compare the current decoder state with all encoder outputs
2. Assign importance scores (attention weights)
3. Compute a weighted combination of encoder outputs

This produces a context vector, which represents the most relevant parts of the input for that step.

⸻

Attention Computation (What Actually Happens)

Given:

* Encoder outputs (all time steps)
* Current decoder hidden state

The model computes:

1. Similarity Scores

decoder state ⋅ encoder outputs

This is a dot product measuring how aligned the decoder is with each input token.

⸻

2. Attention Weights

softmax(similarity scores)

This converts scores into probabilities:

* Higher score → more focus on that word
* Lower score → less focus

⸻

3. Context Vector

weighted sum of encoder outputs

This combines all encoder states, scaled by their importance.

⸻

Combining Information

The model does not rely only on attention.

Instead, it combines:

context vector + decoder hidden state

This combined representation is passed through a linear layer to produce predictions.

⸻

Step-by-Step Flow

Example:

Input: "let's go"

Encoding

* Each word is processed by the encoder LSTM
* All intermediate states are stored

⸻

Decoding (Step 1)

<EOS> → ?

* Decoder generates a hidden state
* Attention compares this state with all input words
* Model focuses on the most relevant parts
* Predicts: vamos

⸻

Decoding (Step 2)

vamos → ?

* Attention recalculates (focus may shift)
* Model predicts: <EOS>

⸻

Key Observation

Attention is recomputed at every step, so focus can change dynamically.

⸻

My Experiment

I used a small dataset:

* “let’s go” → “vamos”
* “to go” → “ir”

Goal:

* Observe how attention distributes focus
* Compare behavior with and without attention

⸻

Observations

What improved

* Model no longer relies only on final encoder state
* Can use information from all input positions
* More flexible during decoding

⸻

What remains limited

* With very small data, behavior is still mostly memorization
* Attention helps structure, but does not create meaning
* Performance depends heavily on training data

⸻

Key Insight

Attention changes the model from:

single compressed representation → dynamic information retrieval

Instead of remembering everything in one vector, the model:

stores everything and selectively retrieves what it needs

⸻

Important Conceptual Shift

Without attention:

Encoder → one fixed summary → Decoder

With attention:

Encoder → all hidden states → Decoder chooses what to use

This removes the bottleneck and makes the model more scalable.

⸻

Final Understanding

* Encoder produces representations for every input token
* Decoder generates output step-by-step
* Attention dynamically selects relevant input information
* Predictions depend on both:
    * current decoder state
    * selected encoder information

⸻

Takeaway

* Attention allows the model to focus on relevant parts of input
* It removes the limitation of fixed-size encoding
* It improves flexibility and scalability of Seq2Seq models
* Learning still depends entirely on patterns in data

⸻

This sets the foundation for more advanced architectures, where attention becomes the central mechanism rather than a supporting component.



🧠 Step-by-Step Flow (Very Clear)

Step 1 — Encoder

* Input tokens → embeddings
* LSTM processes sequence
* Outputs:
    * encoder_outputs → all time steps
    * encoder_hidden, encoder_cell → final summary

⸻

Step 2 — Decoder Start

* Start with <EOS>
* Initialize with encoder memory

⸻

Step 3 — Attention (Core Idea)

Instead of using only final encoder state:

👉 Decoder looks at all encoder outputs

Process:

1. Compare decoder state with every encoder state
2. Get similarity scores
3. Apply softmax → attention weights
4. Compute weighted sum → context vector

⸻

Step 4 — Combine Information

context vector + decoder output → Linear → logits

So prediction uses:

* current decoding state
* relevant parts of input

⸻

Step 5 — Loop

Repeat:

* pick next token
* run decoder
* recompute attention
* generate next word

⸻

🔥 Key Insight (Attention vs No Attention)

Without attention:

decoder uses only final encoder state

With attention:

decoder dynamically looks at ALL input words

⸻


🚨 Final Intuition

* Encoder = stores all word representations
* Attention = decides which words matter now
* Decoder = generates output step-by-step
